# Indexing: Embedding + FAISS (KUHP RAG)

Input: `data/processed/chunks.json` (624 pasal, hasil dari `pre-processing.ipynb`).
Output: `data/index/baseline/` — `embeddings.npy`, `faiss.index`, `config.json`.

Versi ini pakai `text_for_embedding` polos (tanpa breadcrumb Buku/Bab/Bagian) sebagai baseline. Versi breadcrumb dibangun terpisah nanti sebagai pembanding, setelah baseline ini tervalidasi.

In [ ]:
# load data & cek batas token:

import json
from transformers import AutoTokenizer

with open("data/processed/chunks.json", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"Jumlah pasal dimuat: {len(chunks)}")

tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")
panjang_token = [(c["pasal"], len(tokenizer.encode(c["text"]))) for c in chunks]
panjang_token.sort(key=lambda x: x[1], reverse=True)

print("5 pasal dengan token terbanyak:")
for pasal, n_token in panjang_token[:5]:
    print(f"  Pasal {pasal}: {n_token} token")

lebih_dari_batas = [p for p, n in panjang_token if n > 8192]
print(f"Pasal yang melebihi 8192 token: {lebih_dari_batas}")

Jumlah pasal dimuat: 624


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

5 pasal dengan token terbanyak:
  Pasal 622: 5155 token
  Pasal 609: 719 token
  Pasal 473: 604 token
  Pasal 607: 479 token
  Pasal 85: 463 token
Pasal yang melebihi 8192 token: []


In [ ]:
# text_for_embedding (baseline, polos):

for c in chunks:
    c["text_for_embedding"] = c["text"]

print("text_for_embedding disiapkan (baseline, identik dengan text)")

text_for_embedding disiapkan (baseline, identik dengan text)


In [3]:
# load model & encode

from FlagEmbedding import BGEM3FlagModel

model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=False)  # fp16 cuma berguna di GPU, tidak di Mac

texts = [c["text_for_embedding"] for c in chunks]
hasil = model.encode(texts, batch_size=12, max_length=8192)
vecs = hasil["dense_vecs"].astype("float32")

print(f"Shape embeddings: {vecs.shape}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


pre tokenize: 100%|██████████| 52/52 [00:00<00:00, 527.83it/s]


















































Inference Embeddings: 100%|██████████| 52/52 [00:59<00:00,  1.15s/it]

Shape embeddings: (624, 1024)


In [4]:
# bangun FAISS index

import faiss

faiss.normalize_L2(vecs)  # defensif — dense_vecs BGE-M3 sebenarnya sudah ternormalisasi

index = faiss.IndexFlatIP(vecs.shape[1])
index.add(vecs)

print(f"Jumlah vektor di index: {index.ntotal}")

Jumlah vektor di index: 624


In [5]:
# simpan hasil

import os
import numpy as np

os.makedirs("data/index/baseline", exist_ok=True)

np.save("data/index/baseline/embeddings.npy", vecs)
faiss.write_index(index, "data/index/baseline/faiss.index")

config = {
    "model": "BAAI/bge-m3",
    "dim": int(vecs.shape[1]),
    "max_length": 8192,
    "text_for_embedding": "baseline (polos, identik dengan text)",
    "jumlah_pasal": len(chunks),
}
with open("data/index/baseline/config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("Tersimpan ke data/index/baseline/")

Tersimpan ke data/index/baseline/


## BM25

Index BM25 dari `chunks.json` yang sama, pakai `rank_bm25`. Tokenisasi baseline: lowercase + regex kata, tanpa stemming/stopword removal.

In [6]:
# tokenisasi

import re

def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", text.lower())

corpus_tokens = [tokenize(c["text"]) for c in chunks]
print(f"Contoh tokenisasi Pasal {chunks[0]['pasal']}:")
print(corpus_tokens[0][:15])

Contoh tokenisasi Pasal 1:
['pasal', '1', '1', 'tidak', 'ada', 'satu', 'perbuatan', 'pun', 'yang', 'dapat', 'dikenai', 'sanksi', 'pidana', 'dan', 'atau']


In [7]:
# bangun index & tes cepat

from rank_bm25 import BM25Okapi

bm25 = BM25Okapi(corpus_tokens)
print(f"BM25 index dibangun dari {len(corpus_tokens)} pasal")

query = tokenize("pencurian dengan kekerasan")
scores = bm25.get_scores(query)
top5 = scores.argsort()[::-1][:5]

print("\nTes cepat query 'pencurian dengan kekerasan':")
for idx in top5:
    print(f"  Pasal {chunks[idx]['pasal']:3d} (skor {scores[idx]:.2f}) | {chunks[idx]['bab']}")

BM25 index dibangun dari 624 pasal

Tes cepat query 'pencurian dengan kekerasan':
  Pasal 245 (skor 11.50) | BAB V - TINDAK PIDANA TERHADAP KETERTIBAN UMUM
  Pasal 477 (skor 9.74) | BAB XXIV - TINDAK PIDANA PENCURIAN
  Pasal 479 (skor 9.65) | BAB XXIV - TINDAK PIDANA PENCURIAN
  Pasal 476 (skor 7.99) | BAB XXIV - TINDAK PIDANA PENCURIAN
  Pasal 414 (skor 7.01) | BAB XV - TINDAK PIDANA KESUSILAAN


In [8]:
# simpan

import pickle

os.makedirs("data/index/bm25", exist_ok=True)
with open("data/index/bm25/bm25.pkl", "wb") as f:
    pickle.dump(bm25, f)

config_bm25 = {
    "metode": "BM25Okapi (rank_bm25)",
    "tokenisasi": "lowercase + regex [a-z0-9]+ (tanpa stopword removal / stemming)",
    "jumlah_pasal": len(chunks),
}
with open("data/index/bm25/config.json", "w", encoding="utf-8") as f:
    json.dump(config_bm25, f, ensure_ascii=False, indent=2)

print("Tersimpan ke data/index/bm25/")

Tersimpan ke data/index/bm25/


In [9]:
import sys
sys.path.append("src")

import json, pickle, faiss
from FlagEmbedding import BGEM3FlagModel
from retrieval import search

with open("data/processed/chunks.json", encoding="utf-8") as f:
    chunks = json.load(f)
with open("data/index/bm25/bm25.pkl", "rb") as f:
    bm25 = pickle.load(f)
index = faiss.read_index("data/index/baseline/faiss.index")
model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=False)
pasal_index = {c["pasal"]: c for c in chunks}

def tampilkan(hasil):
    for chunk, skor in hasil:
        print(f"  Pasal {chunk['pasal']:3d} (skor {skor:.4f}) | {chunk['bab']}")

# tes 1: routing (harusnya langsung lookup, skip semua retrieval)
print("=== tes routing ===")
tampilkan(search("Pasal 362 itu bagian apa", method="semantic", chunks=chunks,
                  bm25=bm25, model=model, index=index, pasal_index=pasal_index))

# tes 2: query kasus biasa, tanpa reranker dulu (harus sama kayak sebelumnya)
print("\n=== tanpa reranker (hybrid) ===")
tampilkan(search("nyolong motor tetangga itu pasal apa", method="hybrid", chunks=chunks,
                  bm25=bm25, model=model, index=index, pasal_index=pasal_index, top_k=5))

# tes 3: query yang sama, DENGAN reranker — bandingkan urutannya berubah atau tidak
print("\n=== dengan reranker (hybrid) ===")
tampilkan(search("nyolong motor tetangga itu pasal apa", method="hybrid", chunks=chunks,
                  bm25=bm25, model=model, index=index, pasal_index=pasal_index, top_k=5,
                  use_reranker=True))

# tes 4: kasus BM25 yang dulu salah (inget Pasal 245 menang dari Pasal 479?)
# cek apakah reranker berhasil benerin urutan ini
print("\n=== kasus BM25 bermasalah dulu, pakai BM25 + reranker ===")
tampilkan(search("pencurian dengan kekerasan", method="bm25", chunks=chunks,
                  bm25=bm25, model=model, index=index, pasal_index=pasal_index, top_k=5,
                  use_reranker=True))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

=== tes routing ===
  Pasal 362 (skor 1.0000) | BAB IX - TINDAK PIDANA TERHADAP KEKUASAAN PEMERINTAHAN

=== tanpa reranker (hybrid) ===
  Pasal 265 (skor 0.0301) | BAB V - TINDAK PIDANA TERHADAP KETERTIBAN UMUM
  Pasal 209 (skor 0.0265) | BAB I - TINDAK PIDANA TERHADAP KEAMANAN NEGARA
  Pasal 312 (skor 0.0241) | BAB VIII - TINDAK PIDANA YANG MEMBAHAYAKAN KEAMANAN UMUM BAGI ORANG, KESEHATAN, DAN BARANG
  Pasal 461 (skor 0.0231) | BAB XXI - TINDAK PIDANA TERHADAP NYAWA DAN JANIN
  Pasal 454 (skor 0.0226) | BAB XIX - TINDAK PIDANA TERHADAP KEMERDEKAAN ORANG

=== dengan reranker (hybrid) ===


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

  Pasal 265 (skor -3.3509) | BAB V - TINDAK PIDANA TERHADAP KETERTIBAN UMUM
  Pasal 476 (skor -3.4074) | BAB XXIV - TINDAK PIDANA PENCURIAN
  Pasal 478 (skor -3.4229) | BAB XXIV - TINDAK PIDANA PENCURIAN
  Pasal 461 (skor -3.7255) | BAB XXI - TINDAK PIDANA TERHADAP NYAWA DAN JANIN
  Pasal 486 (skor -4.0074) | BAB XXVI - TINDAK PIDANA PENGGELAPAN

=== kasus BM25 bermasalah dulu, pakai BM25 + reranker ===
  Pasal 479 (skor 3.4772) | BAB XXIV - TINDAK PIDANA PENCURIAN
  Pasal 245 (skor 1.5062) | BAB V - TINDAK PIDANA TERHADAP KETERTIBAN UMUM
  Pasal 477 (skor 0.9081) | BAB XXIV - TINDAK PIDANA PENCURIAN
  Pasal 262 (skor -0.9804) | BAB V - TINDAK PIDANA TERHADAP KETERTIBAN UMUM
  Pasal 543 (skor -1.3265) | BAB XXXI - TINDAK PIDANA PELAYARAN


In [11]:
import importlib, retrieval
importlib.reload(retrieval)
from retrieval import rewrite_query, expand_query

q = "nyolong motor tetangga itu pasal apa"
print("Rewrite:", rewrite_query(q))
print("Expand :", expand_query(q))

Rewrite: Bagaimanakah ketentuan hukum pidana yang mengatur mengenai tindak pidana pencurian sepeda motor milik orang lain?
Expand : nyolong motor tetangga itu pasal apa Pencurian kendaraan bermotor, Pencurian


# =====================================================
# =====================================================
